# Cleaning messy data with pandas

A dental clinic pasted the patient exports of its two old booking systems into one file. You will find what is wrong with it from `info()` and a uniqueness check, fix it with written rules, and catch a bad join before it reaches a report.

About 15 minutes. Run the cells from top to bottom.

## Step 1: install the pandas version the lesson used

The lesson's outputs come from pandas 3.1.1. This cell installs it. Older pandas prints `object` where pandas 3 prints `str`, so the version matters for what you see.

In [1]:
%pip install -q pandas==3.1.1

Note: you may need to restart the kernel to use updated packages.


## Step 2: write the two files

This cell writes the clinic's `patients.csv` (15 rows from systems A and B) and `visits.csv` (this month's 8 visits), then prints the pandas version.

**Look for:** `pandas version: 3.1.1`. If it shows another version, use Runtime > Restart session, then run this cell again.

In [2]:
import pandas as pd

PATIENTS_CSV = """system,patient_id,name,birth_date,balance
A,1001,Han Jiwoo,1985-02-14,0
A,1002,Oh Seri,1979-11-23,45000
A,1003,Lim Taeho,,12000
A,1004,Choi Mina,1990-07-04,30000
A,1005,Yoon Dabin,2001-05-30,0
A,1006,Kang Hyun,1968-09-12,8000
A,1007,Seo Yuna,1995-12-01,150000
A,1008,Jang Woo,1983-04-17,0
A,1009,Moon Ara,1972-08-08,22000
B,1010,Baek Sol,03/06/1999,"18,000"
B,1004,Choi Mina,04/07/1990,"30,000"
B,1011,Nam Gyu,21/01/1988,N/A
B,1012,Ryu Hana,15/10/2003,-
B,1013,Shin Bom,30/03/1975,"5,500"
B,1014,Do Eun,09/11/1992,"72,000"
"""

VISITS_CSV = """patient_id,treatment,fee
1001,cleaning,40000
1004,filling,60000
1004,check-up,20000
1007,crown,150000
1010,cleaning,40000
1012,x-ray,25000
1013,check-up,20000
1014,filling,60000
"""

with open("patients.csv", "w") as f:
    f.write(PATIENTS_CSV)
with open("visits.csv", "w") as f:
    f.write(VISITS_CSV)

print("pandas version:", pd.__version__)

pandas version: 3.1.1


## Step 3: load it with no settings and ask `info()`

The same `read_csv` call as in the last lesson. Then `info()` prints each column's type and how many values it has.

**Look for:** `birth_date` and `balance` are `str` (text), each with `14 non-null`. A date column and a money column should not be text. You found two missing balances, but `info()` counts only one: the "-" counted as a value.

In [3]:
patients = pd.read_csv("patients.csv")
print(patients)
print()
patients.info()

   system  patient_id        name  birth_date balance
0       A        1001   Han Jiwoo  1985-02-14       0
1       A        1002     Oh Seri  1979-11-23   45000
2       A        1003   Lim Taeho         NaN   12000
3       A        1004   Choi Mina  1990-07-04   30000
4       A        1005  Yoon Dabin  2001-05-30       0
5       A        1006   Kang Hyun  1968-09-12    8000
6       A        1007    Seo Yuna  1995-12-01  150000
7       A        1008    Jang Woo  1983-04-17       0
8       A        1009    Moon Ara  1972-08-08   22000
9       B        1010    Baek Sol  03/06/1999  18,000
10      B        1004   Choi Mina  04/07/1990  30,000
11      B        1011     Nam Gyu  21/01/1988     NaN
12      B        1012    Ryu Hana  15/10/2003       -
13      B        1013    Shin Bom  30/03/1975   5,500
14      B        1014      Do Eun  09/11/1992  72,000

<class 'pandas.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype
---  

## Step 4: two more checks

Adding up the balances, counting missing values per column, and asking if the key repeats.

**Look for:** the sum is a long glued text that starts `'04500012000`, not a number. `balance` has only 1 missing. `is_unique` is `False`: one patient is in the file twice.

In [4]:
print("sum of balance:", repr(patients["balance"].sum()))
print()
print("missing per column:")
print(patients.isna().sum())
print()
print("is patient_id unique?", patients["patient_id"].is_unique)

sum of balance: '04500012000300000800015000002200018,00030,000-5,50072,000'

missing per column:
system        0
patient_id    0
name          0
birth_date    1
balance       1
dtype: int64

is patient_id unique? False


## Step 5: the balance rule from scratch, in plain Python

No pandas here. The rule: remove spaces at the ends; if the text is empty, "N/A" or "-", the value is missing (`None`); otherwise remove the commas and make it a whole number.

**Look for:** two `None` values (rows 11 and 12), `18000` where the file had "18,000", and a real total of `392500`.

In [5]:
import csv

def clean_balances(raw_balances):
    missing = ("", "N/A", "-")
    cleaned = []
    for raw in raw_balances:
        text = raw.strip()
        if text in missing:
            cleaned.append(None)
            continue
        digits = text.replace(",", "")
        cleaned.append(int(digits))
    return cleaned

with open("patients.csv", newline="") as f:
    rows = list(csv.DictReader(f))
column = [r["balance"] for r in rows]
print("raw texts:  ", column)
cleaned = clean_balances(column)
print("all 15 cleaned:", cleaned)
print("total owed (missing left out):", sum(v for v in cleaned if v is not None))

raw texts:   ['0', '45000', '12000', '30000', '0', '8000', '150000', '0', '22000', '18,000', '30,000', 'N/A', '-', '5,500', '72,000']
all 15 cleaned: [0, 45000, 12000, 30000, 0, 8000, 150000, 0, 22000, 18000, 30000, None, None, 5500, 72000]
total owed (missing left out): 392500


## Step 6: the same rule as two `read_csv` settings

pandas already treats a blank and "N/A" as missing. You add only what is special to this file: "-" is missing too, and a comma separates thousands.

**Look for:** `balance` is now `float64` with `13 non-null`, and its sum is `392500.0`, the same total as your Python rule. The missing values make it `float64`. With `dtype={"balance": "Int64"}` you get whole numbers and `<NA>` for the two missing ones.

In [6]:
patients = pd.read_csv("patients.csv", na_values=["-"], thousands=",")
patients.info()
print()
print("balance sum:", patients["balance"].sum())

p2 = pd.read_csv("patients.csv", na_values=["-"], thousands=",", dtype={"balance": "Int64"})
print("with dtype Int64:", p2["balance"].dtype, p2["balance"].tolist())

<class 'pandas.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   system      15 non-null     str    
 1   patient_id  15 non-null     int64  
 2   name        15 non-null     str    
 3   birth_date  14 non-null     str    
 4   balance     13 non-null     float64
dtypes: float64(1), int64(1), str(3)
memory usage: 732.0 bytes

balance sum: 392500.0
with dtype Int64: Int64 [0, 45000, 12000, 30000, 0, 8000, 150000, 0, 22000, 18000, 30000, <NA>, <NA>, 5500, 72000]


## Step 7: broken on purpose, the dates

`to_datetime` with no format. This cell is meant to fail. The `try` catches the error and prints its message.

**Look for:** a `ValueError` about `"03/06/1999"` and the form `"%Y-%m-%d"`. pandas took that form from the first date in the column (row 0, system A) and used it for every row. Row 9 is the first date written the other way.

In [7]:
try:
    pd.to_datetime(patients["birth_date"])
except Exception as e:
    print(type(e).__name__ + ":", e)

ValueError: time data "03/06/1999" doesn't match format "%Y-%m-%d". You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.


## Step 8: two fixes that run but are wrong

First `format="mixed"`: it guesses each date on its own. Then one format for every row, with `errors="coerce"`, which turns whatever does not match into missing (NaT).

**Look for:** with `mixed`, rows 9, 10 and 14 have day and month swapped (row 10 says 1990-04-07, but row 3, the same patient, says 1990-07-04). With one format, all 9 system A dates become NaT. No error in either case.

In [8]:
mixed = pd.to_datetime(patients["birth_date"], format="mixed")
print("format='mixed':")
print(mixed.to_string())
print()
one = pd.to_datetime(patients["birth_date"], format="%d/%m/%Y", errors="coerce")
print("format='%d/%m/%Y', errors='coerce'   NaT:", one.isna().sum())

format='mixed':
0    1985-02-14
1    1979-11-23
2           NaT
3    1990-07-04
4    2001-05-30
5    1968-09-12
6    1995-12-01
7    1983-04-17
8    1972-08-08
9    1999-03-06
10   1990-04-07
11   1988-01-21
12   2003-10-15
13   1975-03-30
14   1992-09-11

format='%d/%m/%Y', errors='coerce'   NaT: 9


## Step 9: the right fix, each system with its own form

Read the dates once with system A's form and once with system B's form. A date that does not fit becomes NaT for a moment, and `fillna` fills it from the other reading.

**Look for:** exactly `1` NaT (row 2, the real blank), patient 1004 has `1990-07-04` in both rows, and the type is `datetime64`.

In [9]:
dates = patients["birth_date"]
from_a = pd.to_datetime(dates, format="%Y-%m-%d", errors="coerce")
from_b = pd.to_datetime(dates, format="%d/%m/%Y", errors="coerce")
patients["birth_date"] = from_a.fillna(from_b)
print(patients[["system", "patient_id", "birth_date"]].to_string())
print("NaT:", patients["birth_date"].isna().sum(), "  dtype:", patients["birth_date"].dtype)

   system  patient_id birth_date
0       A        1001 1985-02-14
1       A        1002 1979-11-23
2       A        1003        NaT
3       A        1004 1990-07-04
4       A        1005 2001-05-30
5       A        1006 1968-09-12
6       A        1007 1995-12-01
7       A        1008 1983-04-17
8       A        1009 1972-08-08
9       B        1010 1999-06-03
10      B        1004 1990-07-04
11      B        1011 1988-01-21
12      B        1012 2003-10-15
13      B        1013 1975-03-30
14      B        1014 1992-11-09
NaT: 1   dtype: datetime64[us]


## Step 10: the patient who is there twice

`drop_duplicates()` drops a row only if every column is equal. With `subset`, it compares only the columns you name.

**Look for:** the counts `0`, `0`, `1`. The raw rows differ as text. After the fixes they still differ in `system`. With `subset`, one row goes. The last table shows both rows of patient 1004, so you can check that they agree.

In [10]:
raw = pd.read_csv("patients.csv")
cols = ["patient_id", "name", "birth_date", "balance"]
print("raw file, drop_duplicates() removes:", len(raw) - len(raw.drop_duplicates()))
print("after the fixes, drop_duplicates() removes:", len(patients) - len(patients.drop_duplicates()))
print("after the fixes, drop_duplicates(subset=cols) removes:", len(patients) - len(patients.drop_duplicates(subset=cols)))
print()
print(patients[patients["patient_id"].duplicated(keep=False)].to_string())

raw file, drop_duplicates() removes: 0
after the fixes, drop_duplicates() removes: 0
after the fixes, drop_duplicates(subset=cols) removes: 1

   system  patient_id       name birth_date  balance
3       A        1004  Choi Mina 1990-07-04  30000.0
10      B        1004  Choi Mina 1990-07-04  30000.0


## Step 11: broken on purpose, the monthly report

A colleague merged visits onto patients before the duplicate was removed. Then the same merge with `validate="many_to_one"`, which checks that each patient has one row. That second merge is meant to fail.

**Look for:** `report rows: 10` from 8 visits, fees `495000` instead of `415000`, and Choi Mina at `160000`. Then a `MergeError` that names patient 1004.

In [11]:
visits = pd.read_csv("visits.csv")
print("visits rows:", len(visits), "  fees billed:", visits["fee"].sum())

report = visits.merge(patients, on="patient_id", how="left")
print("report rows:", len(report), "  fees in report:", report["fee"].sum())
print(report.groupby("name")["fee"].sum().to_string())
print()
try:
    visits.merge(patients, on="patient_id", how="left", validate="many_to_one")
except Exception as e:
    print(type(e).__name__ + ":", e)

visits rows: 8   fees billed: 415000
report rows: 10   fees in report: 495000
name
Baek Sol      40000
Choi Mina    160000
Do Eun        60000
Han Jiwoo     40000
Ryu Hana      25000
Seo Yuna     150000
Shin Bom      20000

MergeError: Merge keys are not unique in right dataset; not a many-to-one merge

Duplicates in right:
  patient_id
       1004 ...


## Step 12: remove the duplicate, then merge with the check

**Look for:** `patients rows: 14`, `id unique: True`, `validate passed`, `report rows: 8`, fees `415000`, and Choi Mina at `80000`.

In [12]:
once = patients.drop_duplicates(subset=["patient_id", "name", "birth_date", "balance"])
print("patients rows:", len(once), "  id unique:", once["patient_id"].is_unique)
report = visits.merge(once, on="patient_id", how="left", validate="many_to_one")
print("validate passed")
print("report rows:", len(report), "  fees in report:", report["fee"].sum())
print(report.groupby("name")["fee"].sum().to_string())

patients rows: 14   id unique: True
validate passed
report rows: 8   fees in report: 415000
name
Baek Sol      40000
Choi Mina     80000
Do Eun        60000
Han Jiwoo     40000
Ryu Hana      25000
Seo Yuna     150000
Shin Bom      20000


## Now change it

Each cell below has one line marked `# EDIT`. Change it, run the cell, and compare with what it printed before.

**Experiment 1.** Change the marked line to `markers = []`, an empty list of extra markers. `balance` should go back to `str` with 14 non-null values: pandas does not know "-" means missing, so the column cannot be a number.

In [13]:
markers = ["-"]  # EDIT: try []
test = pd.read_csv("patients.csv", na_values=markers, thousands=",")
print("balance dtype:", test["balance"].dtype, "  non-null:", test["balance"].notna().sum())

balance dtype: float64   non-null: 13


**Experiment 2.** Change `keep="first"` to `keep=False`. Now both rows of patient 1004 go, not one. The merge still passes `validate` and still has 8 rows, but Choi Mina disappears from the totals (they add up to `335000`): her two visits have no patient row any more. A check that passes is not proof the data is right.

In [14]:
how_to_keep = "first"  # EDIT: try False
once = patients.drop_duplicates(subset=["patient_id", "name", "birth_date", "balance"], keep=how_to_keep)
report = visits.merge(once, on="patient_id", how="left", validate="many_to_one")
by_name = report.groupby("name")["fee"].sum()
print("patients rows:", len(once), "  report rows:", len(report))
print("sum of per-patient totals:", by_name.sum())
print(by_name.to_string())

patients rows: 14   report rows: 8
sum of per-patient totals: 415000
name
Baek Sol      40000
Choi Mina     80000
Do Eun        60000
Han Jiwoo     40000
Ryu Hana      25000
Seo Yuna     150000
Shin Bom      20000


## What you should see

Tick these off against the outputs above:

- After a plain `pd.read_csv("patients.csv")`, `patients.info()` lists `birth_date` and `balance` as `str`, each with 14 non-null values.
- `patients["balance"].sum()` returns a long text starting `'04500012000`, not a number.
- With `na_values=["-"]` and `thousands=","`, `balance` is `float64` with 13 non-null values, and its sum is `392500.0`.
- `pd.to_datetime(patients["birth_date"])` raises a `ValueError` about "03/06/1999"; reading each system with its own `format` leaves exactly 1 NaT.
- The merge before removing the duplicate gives 10 rows from 8 visits; after `drop_duplicates(subset=...)` it gives 8, and `validate="many_to_one"` passes.